In [7]:
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import pandas as pd
from scipy import ndimage as ndi
from skimage.transform import rescale
from skimage.measure import marching_cubes, mesh_surface_area, label, regionprops, regionprops_table
from skimage import util, measure
import napari
from tifffile import imread
from liffile import LifFile
from pathlib import Path
import imagej
from imagej import Mode
import scyjava as sj
from matplotlib.colors import to_rgba
import tifffile as tiff

# import warnings
# warnings.filterwarnings("ignore")


In [8]:
def step(axis, xa):
    if axis not in xa.coords or xa.coords[axis].size < 2:
        return None
    return float(xa.coords[axis][1] - xa.coords[axis][0])

In [ ]:
folder = Path(r"Z:\Bel\Vascumap_Example_Lifs\EXTRA_LIFS_PLEASE_ADD_HERE\farid")
tif_files = list(folder.glob("*.tif"))
output_dir = r"Z:\Bel\Vascumap_Example_Lifs\max_projections\batch2"
for i, tif_path in enumerate(tif_files):
    tif_name = "".join(os.path.basename(tif_path).lower().replace(".tif", "")).replace(".", "_")
    print(tif_name)
    all_outputs = []

    try:      
        img = imread(tif_path)
        print(img.shape)

        # Extract metadata from the TIFF file
        with tiff.TiffFile(tif_path) as tif:
            metadata = tif.pages[0].tags
            x_resolution = metadata['XResolution'].value if 'XResolution' in metadata else None
            y_resolution = metadata['YResolution'].value if 'YResolution' in metadata else None

        if x_resolution and y_resolution:
            x_um = 1 / x_resolution[0] * x_resolution[1] * 1e6  # Convert to micrometers
            y_um = 1 / y_resolution[0] * y_resolution[1] * 1e6  # Convert to micrometers
        else:
            x_um = y_um = None

        print(f"Pixel size: {x_um} µm x {y_um} µm")

        max_proj = np.max(img, axis=0)  # Z projection
        z_proj = np.max(img, axis=1)  # Y projection

        out_path_max = os.path.join(
            output_dir,
            f"{tif_name}_img{i}_maxproj.ome.tif"
        )
        out_path_z = os.path.join(
            output_dir,
            f"{tif_name}_img{i}_zproj.ome.tif"
        )

        tiff.imwrite(
            out_path_max,
            max_proj.astype(np.float32),
            ome=True,
            metadata={
                "PhysicalSizeX": x_um,
                "PhysicalSizeY": y_um,
                "PhysicalSizeXUnit": "µm",
                "PhysicalSizeYUnit": "µm",
            }
        )

    except Exception as e:
        print(f"Could not process image {i} in {tif_name}: {e}")

merged_stellaris_dorota_lif - r 1 (2)_merged_001 - c=1
(20, 2357, 1445)
Pixel size: 2274955.695237835 µm x 2274955.695237835 µm
merged_stellaris_dorota_lif - r 1_merged_002 - c=1
(9, 2338, 1441)
Pixel size: 2263898.0702532846 µm x 2263903.1954993606 µm
merged_stellaris_dorota_lif - r 1_merged_003 - c=1
(12, 2335, 1442)
Pixel size: 2274955.695237835 µm x 2274955.695237835 µm
merged_stellaris_dorota_lif - r 1_merged_004 - c=1
(12, 2354, 1444)
Pixel size: 2274955.695237835 µm x 2274955.695237835 µm
merged_stellaris_dorota_lif - r 1 (3)_merged - c=1
(24, 2357, 1444)
Pixel size: 2274955.695237835 µm x 2274955.695237835 µm
merged_stellaris_dorota_lif - r 1_merged_000 - c=1
(20, 2352, 1442)
Pixel size: 2278651.767322311 µm x 2278651.767322311 µm
merged_stellaris_dorota_lif - r 1_merged - c=1
(16, 2348, 1442)
Pixel size: 2271570.8366649705 µm x 2271570.8366649705 µm
merged_stellaris_dorota_lif - r 1 (2)_merged - c=1
(21, 2350, 1442)
Pixel size: 2271570.8366649705 µm x 2271570.8366649705 µm
mer

In [16]:
folder = Path(r"Z:\Bel\Vascumap_Example_Lifs\EXTRA_LIFS_PLEASE_ADD_HERE\farid")
lif_files = list(folder.glob("*.lif"))
output_dir = r"Z:\Bel\Vascumap_Example_Lifs\max_projections\unseen_images"
for lif_path in lif_files:
    lif_name = "".join(os.path.basename(lif_path).lower().replace(".lif", "")).replace(".", "_")
    print(lif_name)
    all_outputs = []
    with LifFile(lif_path) as lif:
        number_of_lifs = len(lif.images)

        for i in range(number_of_lifs):
            print("{}, {}".format(i, lif_name))   
            try:      
                img = lif.images[i]
                print(img.dims)
                if img.dims == ('C', 'Z', 'Y', 'X'):
                    print("image shape is {}".format(img.shape))
                    image = img.asarray()
                    image = image[0, :, :, :]  # Take first channel
                    print("image shape is {}".format(image.shape))

                    max_proj = np.max(image, axis=0)
                    xa = img.asxarray()

                    # liffile coordinates are typically in meters → convert to µm
                    x_um = step("X", xa) * 1e6 if step("X", xa) is not None else None
                    print(f"step size um {x_um}")
                    out_path = os.path.join(
                        output_dir,
                        f"{lif_name}_img{i}_maxproj.ome.tif"
                    )

                    tiff.imwrite(
                        out_path,
                        max_proj.astype(np.float32),
                        ome=True,
                        metadata={
                            "PhysicalSizeX": x_um,
                            "PhysicalSizeY": x_um,
                            "PhysicalSizeXUnit": "µm",
                            "PhysicalSizeYUnit": "µm",
                        }
                    )
                if img.dims == ('Z', 'Y', 'X'):
                    image = img.asarray()

                    max_proj = np.max(image, axis=0)
                    xa = img.asxarray()

                    # liffile coordinates are typically in meters → convert to µm
                    x_um = step("X", xa) * 1e6 if step("X", xa) is not None else None
                    print(f"step size um {x_um}")
                    out_path = os.path.join(
                        output_dir,
                        f"{lif_name}_img{i}_maxproj.ome.tif"
                    )

                    tiff.imwrite(
                        out_path,
                        max_proj.astype(np.float32),
                        ome=True,
                        metadata={
                            "PhysicalSizeX": x_um,
                            "PhysicalSizeY": x_um,
                            "PhysicalSizeXUnit": "µm",
                            "PhysicalSizeYUnit": "µm",
                        }
                    )
            except:
                print(f"Could not process image {i} in {lif_name}")


18_07_25
0, 18_07_25
('C', 'Z', 'Y', 'X')
image shape is (2, 38, 8357, 3774)
image shape is (38, 8357, 3774)
step size um 1.3
1, 18_07_25
('C', 'Z', 'Y', 'X')
image shape is (2, 38, 8357, 3774)
image shape is (38, 8357, 3774)
step size um 1.3
2, 18_07_25
('C', 'Z', 'Y', 'X')
image shape is (2, 42, 8357, 3774)
image shape is (42, 8357, 3774)
step size um 1.3
3, 18_07_25
('C', 'Z', 'Y', 'X')
image shape is (2, 42, 8359, 3774)
image shape is (42, 8359, 3774)
step size um 1.3
4, 18_07_25
('C', 'Z', 'Y', 'X')
image shape is (2, 41, 8357, 3774)
image shape is (41, 8357, 3774)
step size um 1.3
5, 18_07_25
('C', 'Z', 'Y', 'X')
image shape is (2, 45, 8355, 3773)
image shape is (45, 8355, 3773)
step size um 1.3
6, 18_07_25
('C', 'Z', 'Y', 'X')
image shape is (2, 43, 8357, 3773)
image shape is (43, 8357, 3773)
step size um 1.3
7, 18_07_25
('C', 'Z', 'Y', 'X')
image shape is (2, 42, 8359, 3774)
image shape is (42, 8359, 3774)
step size um 1.3
8, 18_07_25
('C', 'Z', 'Y', 'X')
image shape is (2, 40,

In [17]:
from pathlib import Path
import numpy as np
import tifffile as tiff
from skimage.transform import downscale_local_mean

in_dir = Path(r"Z:\Bel\Vascumap_Example_Lifs\max_projections\unseen_images")
out_root = in_dir / "downsampled4"
img_out = out_root / "images"
lab_out = out_root / "labels"
img_out.mkdir(parents=True, exist_ok=True)
lab_out.mkdir(parents=True, exist_ok=True)

DS = 4  # integer downsample factor

tif_paths = sorted(list(in_dir.glob("*.tif")) + list(in_dir.glob("*.tiff")))

for p in tif_paths:
    try:
        im = tiff.imread(p)

        if im.ndim != 2:
            raise ValueError(f"{p.name}: expected 2D, got shape {im.shape}")

        h, w = im.shape
        # crop so dimensions are divisible by DS (required for downscale_local_mean)
        h2, w2 = (h // DS) * DS, (w // DS) * DS
        im_c = im[:h2, :w2]

        im_ds = downscale_local_mean(im_c, (DS, DS))

        # cast back to original dtype if you want (common: uint16)
        if np.issubdtype(im.dtype, np.integer):
            im_ds = np.clip(np.rint(im_ds), 0, np.iinfo(im.dtype).max).astype(im.dtype)
        else:
            im_ds = im_ds.astype(np.float32)

        out_img_path = img_out / p.name
        tiff.imwrite(out_img_path, im_ds)

        lab = np.zeros(im_ds.shape, dtype=np.uint8)
        out_lab_path = lab_out / f"{p.stem}_labels.tif"
        tiff.imwrite(out_lab_path, lab)
    except Exception as e:
        print(f"Could not process image {p.name}: {e}")

print(f"Processed {len(tif_paths)} images")
print(f"Downsampled images: {img_out}")
print(f"Blank labels:       {lab_out}")


Processed 55 images
Downsampled images: Z:\Bel\Vascumap_Example_Lifs\max_projections\unseen_images\downsampled4\images
Blank labels:       Z:\Bel\Vascumap_Example_Lifs\max_projections\unseen_images\downsampled4\labels
